<a href="https://colab.research.google.com/github/reezyhudson865/AI_Projects/blob/main/Chatbot%20Creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### AI 102 Final- Skincare Recommendation Chatbot

This Colab notebook follows the creation of a recommendation skincare bot for a final project in my AI 102 class at UT-Knoxville. I modeled the project after the steps of the NLP pipeline. The dataset for the project was found via Kaggle. The project follows the cleaning process, dictionary creation, TF-IDF, Cosine Similarity, and chatbot prompting.

In [ ]:
#code block 1
#import libraries
import pandas as pd
import numpy as np

In [ ]:
#code block 2
#mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#code block 3
#establish csv in colab
file_path = "/content/drive/MyDrive/AI102 Final/skincare_products_clean.csv"

In [ ]:
#code block 4
#read initial csv
df = pd.read_csv(file_path)
df.head()

,product_name,product_url,product_type,clean_ingreds,price
0,The Ordinary Natural Moisturising Factors + HA...,https://www.lookfantastic.com/the-ordinary-nat...,Moisturiser,"['capric triglyceride', 'cetyl alcohol', 'prop...",£5.20
1,CeraVe Facial Moisturising Lotion SPF 25 52ml,https://www.lookfantastic.com/cerave-facial-mo...,Moisturiser,"['homosalate', 'glycerin', 'octocrylene', 'eth...",£13.00
2,The Ordinary Hyaluronic Acid 2% + B5 Hydration...,https://www.lookfantastic.com/the-ordinary-hya...,Moisturiser,"['sodium hyaluronate', 'sodium hyaluronate', '...",£6.20
3,AMELIORATE Transforming Body Lotion 200ml,https://www.lookfantastic.com/ameliorate-trans...,Moisturiser,"['ammonium lactate', 'c12-15', 'glycerin', 'pr...",£22.50
4,CeraVe Moisturising Cream 454g,https://www.lookfantastic.com/cerave-moisturis...,Moisturiser,"['glycerin', 'cetearyl alcohol', 'capric trigl...",£16.00


# Phase 1 Checkpoint

In [ ]:
#code block 5
#import libaries
import string
from nltk.corpus import stopwords

This code imports tools needed for text preprocessing. The string library helps remove punctuation, while NLTK stopwords help remove common words that do not add useful meaning. This matters because the chatbot recommends products based on ingredient text, so the ingredient column needs to be cleaned before matching products to skincare concerns.

In [ ]:
#code block 6
#confirm file path
import os
file_name = os.path.basename(file_path)
print(file_name)

skincare_products_clean.csv


In [ ]:
#code block 7
#search for missing values
if df.isna().sum().sum() == 0:
    print("There are no missing values")
else:
    print("There are missing values")

There are no missing values


In [ ]:
#code block 8
#cleaning the ingredient list text
import string
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_words)

df['clean_ingreds'] = df['clean_ingreds'].apply(remove_punctuation)
df['clean_ingreds'] = df['clean_ingreds'].apply(remove_stopwords)

This code removes punctuation and stopwords from the ingredient column. Cleaning the ingredient text makes the matching process more consistent because ingredients are easier to compare when unnecessary punctuation and filler words are removed. Since the chatbot recommends products based on ingredient matches, this preprocessing step improves the accuracy of the recommendation engine.

In [ ]:
#code block 9
#filter for desired product types
df["product_type"] = df["product_type"].str.lower().str.strip()

#Define unwanted product types
remove_types = ["bath salts", "body wash", "bath oil", "eye care", "peel", "mask", "balm", "mist"]

#Filter out
df = df[~df["product_type"].isin(remove_types)]

#Reset index
df = df.reset_index(drop=True)

#Confirm removal
print("Remaining product types:")
print(df["product_type"].unique())

Remaining product types:
['moisturiser' 'serum' 'oil' 'cleanser' 'toner' 'exfoliator']


# Why We Removed These Values
To increase simplicity of the project for only 2 group members, we will be focusing on the basics of facial skin care. Therefore, product types relating to body care or more supplementary facial care have been excluded.

In [ ]:
#code block 10
#cleaning the price column

#Convert price to string
df["price"] = df["price"].astype(str)

# Extract numeric value only
df["price_clean"] = df["price"].str.extract(r'(\d+\.?\d*)')[0]

# Convert to float
df["price_clean"] = pd.to_numeric(df["price_clean"], errors="coerce")

This code converts the price column into text, extracts the numeric price value, and converts that value into a float. This is necessary because product prices may include currency symbols or formatting that prevents numerical comparison. Cleaning the price column allows the chatbot to sort products by price and later group recommendations into budget-friendly, mid-range, and expensive options.

In [ ]:
#code block 11
#determining product currencies

def detect_currency(price):
    price = str(price)
    if "$" in price:
        return "USD"
    elif "€" in price:
        return "EUR"
    elif "£" in price:
        return "GBP"
    else:
        return "UNKNOWN"

df["currency"] = df["price"].apply(detect_currency)

print(df["currency"].value_counts())

currency
GBP    549
Name: count, dtype: int64


This code identifies whether each product price is listed in USD, EUR, GBP, or an unknown currency. Since the dataset may contain products from different regions, prices need to be standardized to USD before recommendations are ranked.

In [ ]:
#code block 12
#converting to USD
conversion_rates = {
    "USD": 1.0,
    "EUR": 1.08,   # example rate
    "GBP": 1.27,   # example rate
    "UNKNOWN": 1.0
}

df["price_usd"] = df.apply(
    lambda row: row["price_clean"] * conversion_rates[row["currency"]],
    axis=1
)

This code converts product prices into U.S. dollars using predefined conversion rates. Standardizing prices into one currency allows the chatbot to compare products fairly. This is important because price is later used to rank products and place them into affordable, mid-range, and higher-priced recommendation groups.

In [ ]:
#code block 13
#finalize standard price column
df["price"] = df["price_usd"]

#Drop helper columns
df = df.drop(columns=["price_clean", "currency", "price_usd"])

#Round prices
df["price"] = df["price"].round(2)

#Check results
print(df[["price"]].head())
print(df["price"].describe())

   price
0   6.60
1  16.51
2   7.87
3  28.58
4  20.32
count    549.000000
mean      33.766576
std       28.352723
min        3.420000
25%       15.880000
50%       25.400000
75%       41.910000
max      228.600000
Name: price, dtype: float64


This code replaces the original price column with the cleaned USD price, removes temporary helper columns, rounds the final prices, and displays summary statistics. This will make the price recommendation part easier.

#Clean Price Column

We converted the price column from currency datatype to numeric data type, and converted all prices into USD currency

In [ ]:
#code block 14
#validating text processing
import nltk
nltk.download('stopwords')

text_col = "clean_ingreds"

#Stopwords
stop_words = set(stopwords.words('english'))

#Checks
lowercase_check = df[text_col].str.contains(r'[A-Z]', na=False).sum() == 0
punct_check = df[text_col].str.contains(f"[{string.punctuation}]", na=False).sum() == 0
stopword_check = df[text_col].apply(
    lambda x: any(word in stop_words for word in str(x).split())
).sum() == 0

print("=== TEXT CLEANING VALIDATION: skincare_products_clean.csv ===")

print("Lowercasing:", "PASSED ✅" if lowercase_check else "FAILED ❌")
print("Punctuation Removal:", "PASSED ✅" if punct_check else "FAILED ❌")
print("Stopword Removal:", "PASSED ✅" if stopword_check else "FAILED ❌")

if lowercase_check and punct_check and stopword_check:
    print("\nDataset text is fully cleaned and preprocessed ✅")
else:
    print("\nDataset text still needs preprocessing ❌")

=== TEXT CLEANING VALIDATION: skincare_products_clean.csv ===
Lowercasing: PASSED ✅
Punctuation Removal: PASSED ✅
Stopword Removal: PASSED ✅

Dataset text is fully cleaned and preprocessed ✅


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


This code checks whether the ingredient text has been properly cleaned by testing for uppercase letters, punctuation, and stopwords. These checks help confirm that the ingredient column is ready for matching.

In [ ]:
print("Cleaned Dataset (after Code Block 14):")
display(df.head(10))

Cleaned Dataset (after Code Block 14):


,product_name,product_url,product_type,clean_ingreds,price
0,The Ordinary Natural Moisturising Factors + HA...,https://www.lookfantastic.com/the-ordinary-nat...,moisturiser,capric triglyceride cetyl alcohol propanediol ...,6.60
1,CeraVe Facial Moisturising Lotion SPF 25 52ml,https://www.lookfantastic.com/cerave-facial-mo...,moisturiser,homosalate glycerin octocrylene ethylhexyl sal...,16.51
2,The Ordinary Hyaluronic Acid 2% + B5 Hydration...,https://www.lookfantastic.com/the-ordinary-hya...,moisturiser,sodium hyaluronate sodium hyaluronate pantheno...,7.87
3,AMELIORATE Transforming Body Lotion 200ml,https://www.lookfantastic.com/ameliorate-trans...,moisturiser,ammonium lactate c1215 glycerin prunus amygdal...,28.58
4,CeraVe Moisturising Cream 454g,https://www.lookfantastic.com/cerave-moisturis...,moisturiser,glycerin cetearyl alcohol capric triglyceride ...,20.32
5,CeraVe Moisturising Lotion 473ml,https://www.lookfantastic.com/cerave-moisturis...,moisturiser,glycerin capric triglyceride cetearyl alcohol ...,19.05
6,CeraVe Facial Moisturising Lotion No SPF 52ml,https://www.lookfantastic.com/cerave-facial-mo...,moisturiser,glycerin capric triglyceride niacinamide cetea...,16.51
7,The Ordinary Natural Moisturizing Factors + HA...,https://www.lookfantastic.com/the-ordinary-nat...,moisturiser,capric triglyceride cetyl alcohol propanediol ...,8.64
8,CeraVe Smoothing Cream 177ml,https://www.lookfantastic.com/cerave-smoothing...,moisturiser,glycerin behentrimonium methosulfate cetearyl ...,15.24
9,Clinique Moisture Surge 72 Hour Moisturiser 75ml,https://www.lookfantastic.com/clinique-moistur...,moisturiser,dimethicon butylene glycol glycerin trisiloxan...,46.99


# What is the dataset and what question are we asking?

The dataset is a cleaned dataset of skincare products from Kaggle. The guiding research question we are asking is, how do we turn this dataset into a interactive chatbot recommendation engine for suggesting skincare products based on user concerns and preferences?

In [ ]:
#code block 15
#extracting unique ingredients
import ast

#Flatten ingredients
all_ingredients = []
for ingred_list in df["clean_ingreds"]:
    # Ensure each item in ingred_list is processed, assuming it's already a list of strings
    if isinstance(ingred_list, list):
        all_ingredients.extend([i.lower().strip() for i in ingred_list])
    elif isinstance(ingred_list, str): # Handle cases where it might still be a string
        all_ingredients.extend([i.lower().strip() for i in ingred_list.split()])

#Unique ingredients
unique_ingredients = sorted(set(all_ingredients))

#Save to CSV
ingredient_df = pd.DataFrame({"ingredient": unique_ingredients})
ingredient_df.to_csv("unique_ingredients.csv", index=False)

print("Unique ingredient count:", len(unique_ingredients))
print(ingredient_df.head(50))

Unique ingredient count: 1992
                   ingredient
0                           1
1           10hydroxydecanoic
2               110decanediol
3                          12
4                         124
5                12hexanediol
6                       14700
7                       15510
8                       15850
9                       15985
10                      16035
11                      17200
12                        184
13                        188
14                      19140
15     1methylhydantoin2imide
16                          2
17                         20
18    22dimethylhydrocinnamal
19  2bromo2nitropropane13diol
20                 2oleamido1
21                          3
22                        338
23               3cyclohexene
24             3hydroxylauryl
25            3octadecanediol
26                    3oethyl
27                          4
28                      42090
29                      45410
30        4tbutylcyclohexanol
31        

This code collects ingredients from the cleaned ingredient column, standardizes them, removes duplicates, and saves the unique ingredient list to a CSV file. This helps us understand which ingredients appear in the dataset. Creating this list supports the chatbot because it allows us to build a concern-to-ingredient dictionary using actual ingredients found in the products.

In [ ]:
#code block 16
#creating ingredient purpose dictionary categories

ingredient_purpose_dict = {
    # Acne / Oil Control
    "salicylic acid": ["acne", "oiliness", "pores"],
    "benzoyl peroxide": ["acne"],
    "niacinamide": ["acne", "oiliness", "pores", "hyperpigmentation"],
    "tea tree": ["acne", "oiliness"],
    "zinc": ["acne", "oiliness"],

    # Hydration / Dryness
    "glycerin": ["dryness", "dehydration"],
    "hyaluronic acid": ["dryness", "dehydration"],
    "sodium hyaluronate": ["dryness", "dehydration"],
    "ceramide": ["dryness", "sensitivity"],
    "squalane": ["dryness", "dehydration"],
    "panthenol": ["dryness", "dehydration", "sensitivity"],

    # Sensitivity / Redness
    "aloe": ["sensitivity", "redness"],
    "centella": ["sensitivity", "redness"],
    "cica": ["sensitivity", "redness"],
    "allantoin": ["sensitivity", "redness", "dryness"],
    "colloidal oatmeal": ["sensitivity", "redness"],

    # Brightening / Pigmentation
    "vitamin c": ["hyperpigmentation", "dark spots", "dullness"],
    "alpha arbutin": ["hyperpigmentation", "dark spots"],
    "kojic acid": ["hyperpigmentation", "dark spots"],
    "licorice root": ["hyperpigmentation", "redness"],

    # Anti-aging
    "retinol": ["anti-aging", "fine lines"],
    "retinal": ["anti-aging", "fine lines"],
    "peptide": ["anti-aging", "fine lines"],
    "bakuchiol": ["anti-aging", "fine lines"],

    # Exfoliation / Texture
    "glycolic acid": ["uneven texture", "dullness"],
    "lactic acid": ["uneven texture", "dullness"],
    "bha": ["acne", "pores", "oiliness"],
    "pha": ["uneven texture", "sensitivity"]
}

This dictionary connects skincare ingredients to the concerns they may help address, such as acne, dryness, oiliness, redness, sensitivity, dullness, and hyperpigmentation. Instead of recommending products randomly, the chatbot uses ingredient purposes to connect user concerns with products that contain relevant ingredients.

In [ ]:
#code block 17
#showcasing all available concerns

available_concerns = sorted({
    concern
    for concerns in ingredient_purpose_dict.values()
    for concern in concerns
})

print(available_concerns)

['acne', 'anti-aging', 'dark spots', 'dehydration', 'dryness', 'dullness', 'fine lines', 'hyperpigmentation', 'oiliness', 'pores', 'redness', 'sensitivity', 'uneven texture']


This code extracts all unique skincare concerns from the ingredient purpose dictionary. The chatbot can use this list to show users which concerns are available to choose from. This improves the user experience by guiding users toward valid inputs and reducing spelling or wording issues.

In [ ]:
#code block 18
#show available skincare concern categories

print("Available skincare concerns:")
for i, concern in enumerate(available_concerns, start=1):
    print(f"{i}. {concern}")


Available skincare concerns:
1. acne
2. anti-aging
3. dark spots
4. dehydration
5. dryness
6. dullness
7. fine lines
8. hyperpigmentation
9. oiliness
10. pores
11. redness
12. sensitivity
13. uneven texture


### Showing Available Concern Options

This code prints the skincare concerns that users can choose from. Providing a menu of concerns makes the chatbot easier to use and prevents invalid user input. This step supports a guided chatbot experience instead of requiring users to guess the exact wording of each concern.


In [ ]:
#code block 19
#collect user concern input

user_input = input("Enter your skincare concerns separated by commas: ")

user_concerns = [
    concern.strip().lower()
    for concern in user_input.split(",")
    if concern.strip()
]

print("You selected:", user_concerns)


### Collecting User Concern Input

This code allows the user to enter one or more skincare concerns separated by commas. The input is cleaned by removing extra spaces and converting everything to lowercase. This makes the chatbot flexible enough to handle multiple concerns while keeping the input format consistent for matching. Placing this step before scoring prevents errors because `user_concerns` must exist before recommendations are generated.


In [ ]:
#code block 20
#create a searchable product text field

def make_search_text(row):
    product_name = str(row["product_name"]).lower()
    product_type = str(row["product_type"]).lower()
    ingredients = str(row["clean_ingreds"]).lower()

    return f"{product_name} {product_type} {ingredients}"

df["search_text"] = df.apply(make_search_text, axis=1)


### Creating a Searchable Product Text Field

This code combines product name, product type, and cleaned ingredients into one searchable text field. This makes the matching process easier because the chatbot can search one combined column instead of checking several separate columns. Including product name, type, and ingredients gives the recommendation engine more context when identifying relevant products.


# **Applying TF-IDF**

In [ ]:
#code block 21
#TF-IDF ingredient weighting

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

ingredient_text = df["clean_ingreds"].fillna("").astype(str)

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=1000
)

tfidf_matrix = tfidf.fit_transform(ingredient_text)
tfidf_feature_names = tfidf.get_feature_names_out()

avg_tfidf_scores = np.asarray(tfidf_matrix.mean(axis=0)).flatten()

tfidf_importance_dict = dict(zip(tfidf_feature_names, avg_tfidf_scores))


### Applying TF-IDF to Ingredient Text

This step introduces TF-IDF (Term Frequency–Inverse Document Frequency) to measure the importance of ingredients across all skincare products. Instead of treating every ingredient equally, TF-IDF assigns higher importance to ingredients that are more distinctive and lower importance to ingredients that appear very frequently across many products.

Using TF-IDF improves the chatbot recommendation engine by allowing it to prioritize more meaningful ingredients when matching products to user concerns. This results in more accurate and relevant product rankings.


In [ ]:
#code block 22
#TF-IDF weighted scoring function

import re

def score_product_with_tfidf(search_text, user_concerns):
    score = 0
    matched_ingredients = []

    for ingredient, concerns in ingredient_purpose_dict.items():
        ingredient_lower = ingredient.lower()

        if re.search(r"\b" + re.escape(ingredient_lower) + r"\b", search_text):
            if any(concern in concerns for concern in user_concerns):
                tfidf_weight = tfidf_importance_dict.get(ingredient_lower, 0)
                score += tfidf_weight
                matched_ingredients.append(ingredient)

    return score, list(set(matched_ingredients))


### TF-IDF Weighted Scoring Function for Product Recommendations

This function enhances the chatbot’s recommendation logic by combining rule-based ingredient matching with TF-IDF weighting. Instead of assigning a fixed score for every matched ingredient, the function increases the score based on the importance of each ingredient as determined by TF-IDF.

The function scans each product’s searchable text and checks whether it contains ingredients associated with the user’s selected skincare concerns. When a match is found, the TF-IDF weight of that ingredient is added to the product’s score. This ensures that more distinctive and informative ingredients contribute more to the final ranking than common ones.

A default weight of 0 is used for ingredients that do not appear in the TF-IDF dictionary, which prevents unmatched TF-IDF terms from artificially increasing a product’s score. The function also tracks which ingredients contributed to the score, allowing the chatbot to provide transparent explanations for its recommendations.


In [ ]:
#code block 23
#apply TF-IDF scoring and create ranked recommendation results

df[["score", "matched_ingredients"]] = df["search_text"].apply(
    lambda text: pd.Series(score_product_with_tfidf(text, user_concerns))
)

recommendations = df[df["score"] > 0].sort_values(
    by=["score", "price"],
    ascending=[False, True]
)


### Applying TF-IDF Scoring and Generating Ranked Recommendations

This step applies the TF-IDF weighted scoring function to every product in the dataset. For each product, the function evaluates how well its ingredients match the user’s selected skincare concerns and assigns a score based on both relevance and ingredient importance.

The resulting scores and matched ingredients are stored as new columns in the dataset, allowing for transparent and interpretable recommendations. Products with a score greater than zero are retained, ensuring that only relevant items are considered.

The recommendations are then sorted by score in descending order to prioritize the most relevant products, and by price in ascending order to favor more affordable options when scores are similar. This ranking strategy ensures that the chatbot delivers recommendations that are both effective and cost-conscious.


# **Cosine Similarity**

In [ ]:
#code block 24
#cosine similarity to detect very similar recommended products

from sklearn.metrics.pairwise import cosine_similarity

# Create TF-IDF vectors for recommended products only
recommended_indices = recommendations.index

recommended_tfidf_matrix = tfidf_matrix[recommended_indices]

cosine_sim_matrix = cosine_similarity(recommended_tfidf_matrix)

cosine_sim_df = pd.DataFrame(
    cosine_sim_matrix,
    index=recommendations["product_name"],
    columns=recommendations["product_name"]
)

cosine_sim_df.head()

In [ ]:
#code block 25
#find products that are highly similar to each other

similar_product_pairs = []

similarity_threshold = 0.80

product_names = recommendations["product_name"].tolist()

for i in range(len(product_names)):
    for j in range(i + 1, len(product_names)):
        similarity_score = cosine_sim_matrix[i, j]

        if similarity_score >= similarity_threshold:
            similar_product_pairs.append({
                "product_1": product_names[i],
                "product_2": product_names[j],
                "cosine_similarity": similarity_score
            })

similar_products_df = pd.DataFrame(similar_product_pairs)

similar_products_df.sort_values(
    by="cosine_similarity",
    ascending=False
).head(20)

In [ ]:
#code block 26
#remove overly similar products from final recommendations

def diversify_recommendations(recommendations, cosine_sim_matrix, threshold=0.80):
    selected_indices = []

    for i in range(len(recommendations)):
        too_similar = False

        for selected_i in selected_indices:
            if cosine_sim_matrix[i, selected_i] >= threshold:
                too_similar = True
                break

        if not too_similar:
            selected_indices.append(i)

    return recommendations.iloc[selected_indices]

diverse_recommendations = diversify_recommendations(
    recommendations.reset_index(drop=True),
    cosine_sim_matrix,
    threshold=0.80
)

diverse_recommendations[
    ["product_name", "product_type", "price", "score", "matched_ingredients"]
].head(10)

### Using Cosine Similarity to Identify Similar Products

This step adds cosine similarity to compare products based on their TF-IDF ingredient vectors. While TF-IDF helps determine which ingredients are important, cosine similarity measures how similar two products are based on the overall ingredient text. This allows the chatbot to identify products that may be nearly duplicates or very close alternatives.

Adding cosine similarity improves the recommendation engine because it helps avoid showing users several products that are too similar to each other. Instead, the chatbot can provide a more diverse set of recommendations while still keeping the products relevant to the user’s skincare concerns.

In [ ]:
#code block 24
#define recommendation display function

def show_recommendations(recommendations, user_concerns, top_n=5):
    print("=" * 70)
    print("SKINCARE PRODUCT RECOMMENDATIONS")
    print("User concerns:", ", ".join(user_concerns))
    print("=" * 70)

    if recommendations.empty:
        print("No matching products found.")
        print("Try selecting fewer concerns or different concerns.")
        return

    top_products = recommendations.head(top_n)

    for rank, (_, row) in enumerate(top_products.iterrows(), start=1):
        print(f"\nRecommendation #{rank}")
        print("-" * 50)
        print("Product Name:", row["product_name"])
        print("Product Type:", row["product_type"])
        print("Price: $", row["price"])
        print("TF-IDF Match Score:", round(row["score"], 4))
        print("Matched Ingredients:", ", ".join(row["matched_ingredients"]))

        print("Why recommended:")
        for ingredient in row["matched_ingredients"]:
            concerns_helped = ingredient_purpose_dict.get(ingredient, [])
            matching_concerns = [
                concern for concern in user_concerns
                if concern in concerns_helped
            ]

            if matching_concerns:
                print(f"- {ingredient} may help with {', '.join(matching_concerns)}")[ ]



### Displaying Chatbot Recommendations with Explanations

This function formats and presents the product recommendations in a clear, user-friendly way, simulating a chatbot response. Instead of returning raw data, it organizes the results into ranked recommendations and displays key details such as product name, type, price, TF-IDF match score, and matched ingredients.

The function also improves transparency by explaining why each product was recommended. It links matched ingredients back to the user’s selected skincare concerns using the ingredient purpose dictionary, helping users understand how each product addresses their needs.


In [ ]:
#code block 25
#display basic recommendations

show_recommendations(recommendations, user_concerns, top_n=5)


### Creating Dynamic Price Tiers

This code uses the actual product prices in the recommendation results to create data-driven price ranges. Instead of choosing random dollar amounts, the chatbot uses quantiles to split products into cheaper, mid-range, and more expensive groups. This makes the price categories more fair because they are based on the dataset’s real price distribution.


In [ ]:
#code block 26
#create dynamic price range cutoffs

low_cutoff = recommendations["price"].quantile(0.33)
high_cutoff = recommendations["price"].quantile(0.66)

print("Cheapest range: $0 to $", round(low_cutoff, 2))
print("Mid-range: $", round(low_cutoff, 2), "to $", round(high_cutoff, 2))
print("Most expensive: above $", round(high_cutoff, 2))


### Assigning Products to Price Ranges

This code labels each recommended product as cheapest, mid-range, or most expensive using the dynamic cutoffs calculated from the recommendation results. Adding price tiers allows the chatbot to return recommendations that fit different budgets. This improves the chatbot because users receive a variety of options instead of only the cheapest or highest-scoring products.


In [ ]:
#code block 27
#assign each recommended product to a price tier

def assign_price_range(price):
    if price <= low_cutoff:
        return "1_cheapest"
    elif price <= high_cutoff:
        return "2_mid_range"
    else:
        return "3_most_expensive"

recommendations["price_range"] = recommendations["price"].apply(assign_price_range)

recommendations[["product_name", "product_type", "price", "price_range", "score"]].head()


### Selecting One Recommendation per Price Tier in Each Product Category

This code chooses one top product from each price tier within every skincare product category. The chatbot ranks products by TF-IDF match score first so the recommendations stay relevant to the user’s concerns. Price is used as a secondary sorting factor to keep each option practical and organized.


In [ ]:
#code block 28
#select best product from each price tier within each product category

top3_price_tiers_by_category = (
    recommendations
    .sort_values(
        by=["product_type", "price_range", "score", "price"],
        ascending=[True, True, False, True]
    )
    .groupby(["product_type", "price_range"])
    .head(1)
    .reset_index(drop=True)
)

top3_price_tiers_by_category[
    ["product_type", "price_range", "product_name", "price", "score", "matched_ingredients"]
]


### Cleaning Price Range Labels for User Output

This code converts internal price range labels into clearer user-facing labels. The labels help the chatbot present recommendations as a cheapest option, mid-range option, and most expensive option. This makes the final output easier for users to understand.


In [ ]:
#code block 29
#clean price range labels for user-facing output

price_label_map = {
    "1_cheapest": "Cheapest Option",
    "2_mid_range": "Mid-Range Option",
    "3_most_expensive": "Most Expensive Option"
}

top3_price_tiers_by_category["price_label"] = top3_price_tiers_by_category["price_range"].map(price_label_map)

price_order = ["1_cheapest", "2_mid_range", "3_most_expensive"]
top3_price_tiers_by_category["price_range"] = pd.Categorical(
    top3_price_tiers_by_category["price_range"],
    categories=price_order,
    ordered=True
)

top3_price_tiers_by_category = top3_price_tiers_by_category.sort_values(
    by=["product_type", "price_range"]
)


### Displaying Final Chatbot Recommendations

This function presents the final recommendations in a user-friendly chatbot format. Products are grouped by category, and each category includes a cheapest, mid-range, and most expensive option when available. The output also explains which ingredients matched the user’s skincare concerns, making the recommendation engine more transparent.


In [ ]:
#code block 30
#define final chatbot recommendation output

def show_final_chatbot_recommendations(results, user_concerns):
    print("=" * 80)
    print("FINAL SKINCARE CHATBOT RECOMMENDATIONS")
    print("Based on your concerns:", ", ".join(user_concerns))
    print("=" * 80)

    if results.empty:
        print("No matching products found. Try choosing different skincare concerns.")
        return

    for product_type, group in results.groupby("product_type", observed=False):
        print(f"\n{product_type.upper()}")
        print("-" * 60)

        for i, (_, row) in enumerate(group.iterrows(), start=1):
            print(f"{i}. {row['price_label']}")
            print("Product:", row["product_name"])
            print("Price: $", row["price"])
            print("TF-IDF Match Score:", round(row["score"], 4))
            print("Matched Ingredients:", ", ".join(row["matched_ingredients"]))

            print("Why recommended:")
            for ingredient in row["matched_ingredients"]:
                concerns_helped = ingredient_purpose_dict.get(ingredient, [])

                matching_concerns = [
                    concern for concern in user_concerns
                    if concern in concerns_helped
                ]

                if matching_concerns:
                    print(f"- {ingredient} may help with {', '.join(matching_concerns)}")

            print()


In [ ]:
#code block 31
#run final recommendation output

show_final_chatbot_recommendations(top3_price_tiers_by_category, user_concerns)


## Final Recommendation System Summary

This chatbot recommendation system takes user-selected skincare concerns and matches them to products based on ingredient relevance. The system uses a dictionary that maps ingredients to specific skincare concerns, allowing it to identify which products are most appropriate for the user.

To improve recommendation quality, the system uses TF-IDF weighting so that matched ingredients are not all treated equally. Ingredients that are more distinctive in the dataset contribute more to the score, while very common ingredients contribute less. Products are then ranked by TF-IDF match score to prioritize those that best address the user’s concerns.

In addition to relevance, the system incorporates price as a secondary factor. Products are divided into three price tiers—cheapest, mid-range, and most expensive—using data-driven thresholds based on the recommendation results. This ensures that users receive recommendations across different budget levels.

Overall, this approach combines text preprocessing, domain vocabulary standardization, TF-IDF weighting, rule-based matching, and ranking logic to simulate a practical and interpretable skincare recommendation chatbot.
